# CO-SPY — Fusion V3 Save Version (Reuse Existing Feature Cache)

Notebook này dùng để **Save Version trên Kaggle** sau khi lần chạy trước đã tạo xong feature cache nhưng Fusion V3 crash tại `global_step=0`.

Notebook sẽ:
- dựng lại source vào `/kaggle/working/CO-SPY`;
- patch lỗi Semantic `1153 -> Linear(1152)`;
- nối checkpoint Semantic / Artifact / NPR từ `co-spy-v2`;
- nối trực tiếp feature cache 5+ GB từ `co-spy-fusion-v3-feature-cache`;
- kiểm tra shape cache;
- **dừng ngay** nếu trainer cố cache lại feature;
- train Fusion V3 và lưu checkpoint mới vào `/kaggle/working`.

Kaggle Inputs cần có:
1. `datnguyentrung/co-spy-v2`
2. `datnguyentrung/co-spy-fusion-v3-feature-cache`


In [ ]:
from pathlib import Path
import os
import sys
import shutil
import subprocess
import numpy as np

SOURCE_ROOT = Path(
    "/kaggle/input/datasets/datnguyentrung/"
    "co-spy-fusion-v3-feature-cache/CO-SPY"
)
BRANCH_CKPT_ROOT = Path(
    "/kaggle/input/datasets/datnguyentrung/"
    "co-spy-v2/ckpt/sd-v1_4"
)
CACHE_SRC = SOURCE_ROOT / "ckpt/sd-v1_4/fusion_v3/features/92701a0c35fe"
WORK_ROOT = Path("/kaggle/working/CO-SPY")
CACHE_DST = WORK_ROOT / "ckpt/sd-v1_4/fusion_v3/features/92701a0c35fe"

assert SOURCE_ROOT.exists(), f"Missing source dataset: {SOURCE_ROOT}"
assert BRANCH_CKPT_ROOT.exists(), f"Missing branch checkpoint dataset: {BRANCH_CKPT_ROOT}"
assert CACHE_SRC.exists(), f"Missing saved feature cache: {CACHE_SRC}"

print("SOURCE_ROOT      =", SOURCE_ROOT)
print("BRANCH_CKPT_ROOT =", BRANCH_CKPT_ROOT)
print("CACHE_SRC        =", CACHE_SRC)
print("WORK_ROOT        =", WORK_ROOT)
print("Preflight inputs ✅")


In [ ]:
WORK_ROOT.mkdir(parents=True, exist_ok=True)

SKIP_TOP_LEVEL = {
    "ckpt",
    ".git",
    "data",
    "dataSets",
    "__pycache__",
}

for item in SOURCE_ROOT.iterdir():
    if item.name in SKIP_TOP_LEVEL:
        continue

    target = WORK_ROOT / item.name
    if item.is_dir():
        shutil.copytree(item, target, dirs_exist_ok=True)
    else:
        shutil.copy2(item, target)

fusion_file = WORK_ROOT / "detectors/sd_v1_4/fusion.py"
assert fusion_file.exists(), f"fusion.py not found: {fusion_file}"

print("Source copied ✅")
print("fusion.py =", fusion_file)


In [ ]:
# Install the missing runtime dependency into the SAME Python interpreter used by training.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("loguru") is None:
    print("Installing missing dependency: loguru")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "loguru"],
        check=True,
    )
else:
    print("loguru already installed ✅")

from loguru import logger
print("loguru import ✅", sys.executable)


In [ ]:
p = WORK_ROOT / "detectors/sd_v1_4/fusion.py"
text = p.read_text()

old = "        sem_logit = self.sem.fc(sem_feat)\n"

replacement_lines = [
    "        expected_sem_dim = self.sem.fc.in_features + self.sem.fc.out_features",
    "",
    "        if sem_feat.shape[1] != expected_sem_dim:",
    "            raise ValueError(",
    '                f"Expected semantic fusion features with {expected_sem_dim} dims "',
    '                f"(embedding={self.sem.fc.in_features} + "',
    '                f"logit={self.sem.fc.out_features}), "',
    '                f"got {sem_feat.shape[1]}"',
    "            )",
    "",
    "        # sem_feat already contains [semantic_embedding, semantic_logit]",
    "        sem_logit = sem_feat[:, -self.sem.fc.out_features :]",
]
new = "\n".join(replacement_lines) + "\n"

marker = "sem_feat[:, -self.sem.fc.out_features :]"

if marker in text:
    print("fusion.py already patched ✅")
elif old in text:
    p.write_text(text.replace(old, new, 1))
    print("fusion.py patched ✅")
else:
    raise RuntimeError(
        "Could not find the old semantic-logit line and patch marker is absent. "
        "Stop to avoid training the wrong source."
    )

patched = p.read_text()
assert marker in patched, "Patch verification failed"

subprocess.run(
    [sys.executable, "-m", "py_compile", str(p)],
    check=True,
)

print("FIX FOUND = True")
print("Python syntax check ✅")


In [ ]:
for branch in ["semantic", "artifact", "npr"]:
    src = BRANCH_CKPT_ROOT / branch / "checkpoint_best.pth"
    dst = WORK_ROOT / "ckpt/sd-v1_4" / branch / "checkpoint_best.pth"

    assert src.exists(), f"Missing {branch} checkpoint: {src}"
    dst.parent.mkdir(parents=True, exist_ok=True)

    if dst.exists() or dst.is_symlink():
        dst.unlink()

    os.symlink(src, dst)

    assert dst.exists()
    assert dst.is_symlink()
    print(f"{branch:8} -> {dst}")

print("Branch checkpoints linked ✅")


In [ ]:
CACHE_DST.parent.mkdir(parents=True, exist_ok=True)

if CACHE_DST.exists() or CACHE_DST.is_symlink():
    if CACHE_DST.is_symlink() or CACHE_DST.is_file():
        CACHE_DST.unlink()
    else:
        shutil.rmtree(CACHE_DST)

os.symlink(CACHE_SRC, CACHE_DST, target_is_directory=True)

assert CACHE_DST.exists(), "Cache link target is not readable"
assert CACHE_DST.is_symlink(), "Expected CACHE_DST to be a symlink"

print("SOURCE :", CACHE_SRC)
print("TARGET :", CACHE_DST)
print("SYMLINK:", CACHE_DST.is_symlink())
print("FILES  :", len(list(CACHE_DST.iterdir())))


In [ ]:
expected = {
    "train_sem.npy":    ((191807, 1153), np.float16),
    "train_art.npy":    ((191807, 512),  np.float16),
    "train_npr.npy":    ((191807, 512),  np.float16),
    "train_labels.npy": ((191807,),       np.int8),
    "val_sem.npy":      ((10000, 1153),  np.float16),
    "val_art.npy":      ((10000, 512),   np.float16),
    "val_npr.npy":      ((10000, 512),   np.float16),
    "val_labels.npy":   ((10000,),        np.int8),
}

for name, (shape, dtype) in expected.items():
    path = CACHE_DST / name
    assert path.exists(), f"Missing cached feature file: {path}"

    arr = np.load(path, mmap_mode="r")
    print(f"{name:20} shape={arr.shape} dtype={arr.dtype}")

    assert arr.shape == shape, f"{name}: expected {shape}, got {arr.shape}"
    assert arr.dtype == dtype, f"{name}: expected {dtype}, got {arr.dtype}"

for metadata in ["train_metadata.json", "val_metadata.json"]:
    assert (CACHE_DST / metadata).exists(), f"Missing {metadata}"

print("\nFeature cache validation ✅")
print("Semantic cache = 1152-d embedding + 1-d semantic logit = 1153 ✅")


In [ ]:
fusion_file = WORK_ROOT / "detectors/sd_v1_4/fusion.py"
marker = "sem_feat[:, -self.sem.fc.out_features :]"

print("SOURCE EXISTS =", fusion_file.exists())
print("FIX FOUND     =", marker in fusion_file.read_text())
print("CACHE EXISTS  =", CACHE_DST.exists())
print("CACHE LINK    =", CACHE_DST.is_symlink())

for branch in ["semantic", "artifact", "npr"]:
    p = WORK_ROOT / "ckpt/sd-v1_4" / branch / "checkpoint_best.pth"
    print(f"{branch.upper():8} CKPT =", p.exists(), "| symlink =", p.is_symlink())

assert fusion_file.exists()
assert marker in fusion_file.read_text()
assert CACHE_DST.exists() and CACHE_DST.is_symlink()

print("\nREADY TO TRAIN FUSION V3 ✅")


In [ ]:
# Preflight local imports BEFORE starting the long training process.
import subprocess
import sys

checks = [
    "import dataSets; print('dataSets import ✅')",
    "import evaluate; print('evaluate import ✅')",
]

for snippet in checks:
    subprocess.run(
        [sys.executable, "-c", snippet],
        cwd=str(WORK_ROOT),
        check=True,
    )

print("Local source/import preflight ✅")


In [ ]:
cmd = [
    sys.executable, "-u", "main.py",
    "--gpu", "0",
    "--phase", "train",
    "--mode", "fusion",
    "--train_dataset", "sd-v1_4",
    "--ckpt", str(WORK_ROOT / "ckpt"),
    "--fusion_head", "v3",
    "--epochs", "5",
    "--early-stopping-patience", "2",
    "--artifact_views", "1",
    "--num-workers", "2",
    "--prefetch-factor", "2",
    "--precision", "auto",
]

print("RUNNING:")
print(" ".join(cmd))
print()

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

proc = subprocess.Popen(
    cmd,
    cwd=str(WORK_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

cache_rebuild_detected = False

try:
    assert proc.stdout is not None

    for line in proc.stdout:
        print(line, end="")

        if "Caching fusion features" in line:
            cache_rebuild_detected = True
            print(
                "\n\n[SAFETY STOP] Trainer tried to rebuild the feature cache. "
                "Stopping now so the 30,000+ second extraction is NOT repeated."
            )
            proc.terminate()
            try:
                proc.wait(timeout=15)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
            break

    if cache_rebuild_detected:
        raise RuntimeError(
            "Feature cache was not recognized. Training was intentionally stopped "
            "before re-caching."
        )

    return_code = proc.wait()

    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)

finally:
    if proc.poll() is None:
        proc.terminate()

print("\nFusion V3 training completed ✅")


In [ ]:
fusion_out = WORK_ROOT / "ckpt/sd-v1_4/fusion_v3"

print("Fusion output directory:", fusion_out)
print()

if fusion_out.exists():
    for p in sorted(fusion_out.iterdir()):
        if p.name == "features":
            print("features/  -> reused Input cache")
            continue

        if p.is_file():
            size_mb = p.stat().st_size / 1024**2
            print(f"{p.name:35} {size_mb:10.2f} MB")
        else:
            print(f"{p.name}/")
else:
    print("Fusion output directory not found.")


## Kỳ vọng khi Save Version

- Không chạy lại feature extraction.
- Nếu xuất hiện `Caching fusion features ...`, notebook sẽ **dừng ngay**.
- Fusion V3 bắt đầu training từ epoch 1 vì lần trước chết tại `global_step=0`.
- Checkpoint mới được ghi trong `/kaggle/working/CO-SPY/ckpt/sd-v1_4/fusion_v3`.
